# Module 6: Long-Context Modeling
Comparing LSTMs, Transformers, Linear SSMs, and Selective SSMs on IMDb-Long.


## Part 1: Dataset Loading & IMDb-Long Construction


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import os
import re
import random
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset 

torch.manual_seed(42)
random.seed(42)

print("Loading WikiText-2...")
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
wiki_sentences = []
for item in wiki_dataset:
    text = item['text'].strip()
    if len(text) > 20: 
        wiki_sentences.extend([w for w in re.split(r'[^a-zA-Z0-9]+', text.lower()) if w])

DATA_ROOT = os.path.join('..', 'aclImdb')
def load_imdb(directory, label, max_samples=None):
    data = []
    filenames = sorted([f for f in os.listdir(directory) if f.endswith('.txt')])
    if max_samples: filenames = filenames[:max_samples]
    for fname in filenames:
        with open(os.path.join(directory, fname), 'r', encoding='utf-8', errors='ignore') as f:
            tokens = [t for t in re.split(r'[^a-zA-Z0-9]+', f.read().lower()) if t]
            data.append((tokens, label))
    return data

NUM_SAMPLES = 500
train_pos = load_imdb(os.path.join(DATA_ROOT, 'train', 'pos'), 1, NUM_SAMPLES)
train_neg = load_imdb(os.path.join(DATA_ROOT, 'train', 'neg'), 0, NUM_SAMPLES)
test_pos = load_imdb(os.path.join(DATA_ROOT, 'test', 'pos'), 1, NUM_SAMPLES // 5)
test_neg = load_imdb(os.path.join(DATA_ROOT, 'test', 'neg'), 0, NUM_SAMPLES // 5)
train_data = train_pos + train_neg
test_data = test_pos + test_neg
random.shuffle(train_data)
random.shuffle(test_data)

word_counts = Counter()
for tokens, _ in train_data: word_counts.update(tokens)
word2idx = {'<PAD>': 0, '<UNK>': 1}
for idx, (word, _) in enumerate(word_counts.most_common(4998), start=2):
    word2idx[word] = idx

wiki_idx = [word2idx.get(w, 1) for w in wiki_sentences]

def construct_imdb_long(dataset, target_len):
    long_X, long_Y = [], []
    for tokens, label in dataset:
        idx_seq = [word2idx.get(t, 1) for t in tokens]
        
        thirds = max(1, len(idx_seq) // 3)
        P1 = idx_seq[:thirds]
        # Discard P2
        P3 = idx_seq[-thirds:]
        
        # Sample random filler sentence length from Wiki (e.g. 50 tokens) to replace P2
        filler_len = 50
        start_w = random.randint(0, len(wiki_idx) - filler_len - 1)
        neutral_filler = wiki_idx[start_w : start_w + filler_len]
        
        # P1 || neutral filler || P3
        final_seq = P1 + neutral_filler + P3
        
        # Append additional neutral sentences if needed to reach target length
        if len(final_seq) < target_len:
            missing = target_len - len(final_seq)
            start_w2 = random.randint(0, len(wiki_idx) - missing - 1)
            additional_filler = wiki_idx[start_w2 : start_w2 + missing]
            final_seq += additional_filler
            
        long_X.append(final_seq[:target_len])
        long_Y.append(label)
    
    return torch.tensor(long_X, dtype=torch.long), torch.tensor(long_Y, dtype=torch.long)

print("Constructing targets: L=256, 512, 1024...")
datasets_dict = {}
for L in [256, 512, 1024]:
    tx, ty = construct_imdb_long(train_data, L)
    vx, vy = construct_imdb_long(test_data, L)
    datasets_dict[L] = (torch.utils.data.TensorDataset(tx, ty), torch.utils.data.TensorDataset(vx, vy))
print("Dataset generation complete.")


## Part 5: Model Implementations


### (a) Linear SSM provided skeleton implementation


In [ ]:
class LinearSSM(nn.Module):
    def __init__(self, d_in, d_hidden, num_classes):
        super().__init__()
        self.A = nn.Parameter(torch.randn(d_hidden, d_hidden) * 0.1)
        self.B = nn.Linear(d_in, d_hidden)
        self.C = nn.Linear(d_hidden, d_hidden)
        self.fc = nn.Linear(d_hidden, num_classes)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        
        # Initialize hidden state
        h = torch.zeros(batch, self.A.shape[0], device=x.device)
        
        for t in range(seq_len):
            x_t = x[:, t]
            
            # Linear state recurrence
            # h = A h + B x_t
            h = torch.matmul(h, self.A.t()) + self.B(x_t)
            
            # Project hidden state to output representation
            y = self.C(h)
            
        # Optional: capture final step output out of the loop
        out = self.fc(y)
        return out


### (b) Selective SSM provided skeleton implementation


In [ ]:
class SelectiveSSM(nn.Module):
    def __init__(self, d_in, d_hidden, num_classes):
        super().__init__()
        self.A = nn.Parameter(torch.randn(d_hidden, d_hidden) * 0.1)
        self.B = nn.Linear(d_in, d_hidden)
        self.C = nn.Linear(d_hidden, d_hidden)
        
        # Define input-dependent gating network
        self.W_g = nn.Linear(d_in, d_hidden)
        
        self.fc = nn.Linear(d_hidden, num_classes)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        
        # Initialize hidden state
        h = torch.zeros(batch, self.A.shape[0], device=x.device)
        
        for t in range(seq_len):
            x_t = x[:, t]
            
            # Update recurrent state
            h = torch.matmul(h, self.A.t()) + self.B(x_t)
            
            # Compute selective gate
            g = torch.sigmoid(self.W_g(x_t))
            
            # Apply gate to output projection
            y = g * self.C(h)
            
        # Classify from final sequence representation
        out = self.fc(y)
        return out


### Wrapper Modules (LSTM & Transformer)


In [ ]:

class RNNWrapper(nn.Module):
    def __init__(self, d_in, d_hidden, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(d_in, d_hidden, batch_first=True)
        self.fc = nn.Linear(d_hidden, num_classes)
        
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        # Sequence final state
        return self.fc(h_n[-1])

class TransformerWrapper(nn.Module):
    def __init__(self, d_in, d_hidden, num_classes):
        super().__init__()
        # PyTorch built-in TransformerEncoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_in, nhead=4, dim_feedforward=d_hidden, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_in, num_classes)
        
    def forward(self, x):
        # We don't apply positional encoding here since it's an extreme scaled test
        out = self.transformer(x)
        pooled = torch.mean(out, dim=1) # Mean pooling
        return self.fc(pooled)



### Training Function


In [ ]:

D_EMB = 32
D_HIDDEN = 64
VOCAB_SIZE = 5000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_model(ModelClass, train_dataset, test_dataset, epochs=3):
    model = nn.Sequential(
        nn.Embedding(VOCAB_SIZE, D_EMB),
        ModelClass(D_EMB, D_HIDDEN, 2)
    ).to(DEVICE)
    
    loader_train = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
    loader_test = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    start_time = time.time()
    for ep in range(epochs):
        model.train()
        for x, y in loader_train:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / epochs
    
    model.eval()
    corr = 0
    total = 0
    with torch.no_grad():
        for x, y in loader_test:
            out = model(x.to(DEVICE))
            preds = out.argmax(dim=1)
            corr += (preds == y.to(DEVICE)).sum().item()
            total += y.size(0)
    
    return (corr / total), epoch_time

print("Modules and training logic defined.")



## Part 5c: Experiments & Evaluation Grid (Train all models over lengths [256, 512, 1024])


In [ ]:

models_to_test = {
    'LSTM': RNNWrapper,
    'Transformer': TransformerWrapper,
    'Linear SSM': LinearSSM,
    'Selective SSM': SelectiveSSM
}

lengths = [256, 512, 1024]
results_acc = {m: [] for m in models_to_test}
results_time = {m: [] for m in models_to_test}

for L in lengths:
    print(f"\n--- Context Length: {L} ---")
    train_ds, test_ds = datasets_dict[L]
    for model_name, model_cls in models_to_test.items():
        print(f"Training {model_name}...")
        acc, t_epoch = train_model(model_cls, train_ds, test_ds, epochs=3)
        results_acc[model_name].append(acc)
        results_time[model_name].append(t_epoch)
        print(f"  Accuracy: {acc:.4f} | Time/Epoch: {t_epoch:.2f}s")



### Visualization: Accuracy vs Sequence Length


In [ ]:

# Required to plot Accuracy vs Sequence Length
plt.figure(figsize=(8, 5))
for m in models_to_test:
    plt.plot(lengths, results_acc[m], marker='o', label=m)

plt.xlabel("Sequence Length L")
plt.ylabel("Test Accuracy")
plt.title("Accuracy vs Sequence Length on IMDb-Long")
plt.xticks(lengths)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("q6_accuracy_vs_length.png")
plt.show()

